
# Selective Refinement — General Tree Theorem Gate

This notebook formalizes and stress-tests the theorem suggested by the Global Compatibility Gate.

## Goal

For an arbitrary finite rooted tree, prove that the projection of the full latent trajectory-flow model onto the leaf distribution is exactly a compact polytope in unconditional subtree masses.

The gate has four parts:

1. **Projection / coupling theorem** for taxonomically correct monotone trajectories.
2. **Linear sensitivity theorem** for sibling odds-ratio bounds.
3. **Sharpness corollary** for leaf probabilities and linear risks.
4. **Temporal information theorem**: extra horizons can never widen the identified set, and strict gain is exactly a linear-program redundancy question.

The notebook then stress-tests the theorem on many randomly generated trees and verifies a constructive coupling algorithm.

A pass means the mathematics is ready to be written formally and then checked against the literature. It does **not** by itself establish publication novelty.



## 1. Model and notation

Let \(T\) be a finite rooted tree with leaf set \(\mathcal L\). For each observation:

- \(Y\in\mathcal L\) is the latent true leaf.
- \(C_1,\ldots,C_H\) are observed labels at ordered horizons.
- The trajectory is **monotone**:
  \[
  C_1 \preceq C_2 \preceq \cdots \preceq C_H \preceq Y,
  \]
  where \(u\preceq v\) means that \(u\) is an ancestor of \(v\).

Let
\[
p_\ell=P(Y=\ell),\qquad
s_v=\sum_{\ell\in\mathcal L(v)}p_\ell,
\]
where \(\mathcal L(v)\) denotes leaves below node \(v\).

Let \(V=C_H\) be the final observed node and
\[
M_v=P(V\in \mathrm{subtree}(v)).
\]

For each non-root child \(c\) and horizon \(h\), define the observable crossing mass
\[
r_{c,h}=P(C_h\in \mathrm{subtree}(c)).
\]

Because trajectories are taxonomically correct, crossing edge \(c\) implies \(Y\in\mathcal L(c)\).



## 2. Theorem 1 — Projection of the trajectory-flow polytope

### Theorem

Fix the observed distribution of valid monotone trajectories \(G=(C_1,\ldots,C_H)\).
A leaf distribution \(p\) can be coupled with those trajectories while satisfying
\(C_H\preceq Y\) **if and only if**

\[
p_\ell\ge 0,\qquad \sum_{\ell\in\mathcal L}p_\ell=1,
\]
and, for every non-root node \(v\),
\[
s_v\ge M_v.
\]

Thus the exponentially larger latent trajectory-flow representation is unnecessary for describing the projection onto \(p\).

### Necessity

If \(V\in\mathrm{subtree}(v)\), then \(Y\in\mathcal L(v)\). Hence
\[
M_v=P(V\in\mathrm{subtree}(v))
\le P(Y\in\mathcal L(v))=s_v.
\]

### Sufficiency: constructive laminar coupling

Let
\[
m_v=P(V=v).
\]

Start with leaf capacities \(p_\ell\). Process tree nodes from deepest to root.

When node \(v\) is processed, all supplies \(m_w\) from proper descendants \(w\) have already been assigned to leaves below \(w\). The total residual leaf capacity below \(v\) is

\[
s_v-\sum_{w\in\mathrm{subtree}(v),\,w\ne v}m_w
=
s_v-(M_v-m_v)
\ge m_v.
\]

Therefore all of \(m_v\) can be assigned to still-unfilled leaves below \(v\).
At the root, total supply and total capacity are both one, so all capacities are exactly filled.

This constructs a coupling \(P(V,Y)\) with \(V\preceq Y\).

Finally, conditional on \(V=v\), sample the complete observed trajectory \(G\) according to its **observed** conditional distribution \(P(G\mid V=v)\), independently of \(Y\). Every such trajectory ends at \(v\), and every earlier node is an ancestor of \(v\), so the complete trajectory remains compatible with \(Y\). The original trajectory distribution is reproduced exactly.

Hence the subtree inequalities are sufficient.



## 3. Theorem 2 — Linear sensitivity constraints

For child \(c\) of an internal node and horizon \(h\),

\[
q_{c,h}
=
P(C_h\in\mathrm{subtree}(c)\mid Y\in\mathcal L(c))
=
\frac{r_{c,h}}{s_c}.
\]

For siblings \(j,k\), impose the sensitivity restriction

\[
\frac{q_{j,h}/(1-q_{j,h})}
     {q_{k,h}/(1-q_{k,h})}
\le \Gamma_{jk,h},
\qquad \Gamma_{jk,h}\ge 1.
\]

For interior probabilities this is exactly equivalent to

\[
r_{j,h}s_k
-
\Gamma_{jk,h}r_{k,h}s_j
\le
(1-\Gamma_{jk,h})r_{j,h}r_{k,h}.
\]

The reverse ordered pair gives the lower odds-ratio bound.

These are **linear inequalities in the leaf probabilities**, because every \(s_v\) is a sum of leaf masses.

### Projection result with sensitivity

Let \(\mathcal K\) contain the simplex constraints, all final-node compatibility constraints \(s_v\ge M_v\), and all desired sibling sensitivity half-spaces.

By Theorem 1, every \(p\in\mathcal K\) admits a valid coupling with the complete observed trajectories. The sensitivity probabilities depend only on the fixed crossing masses \(r_{c,h}\) and the subtree masses \(s_c(p)\). Therefore the coupling automatically satisfies every sensitivity bound encoded in \(\mathcal K\).

Hence:

\[
\boxed{
\operatorname{proj}_p(\text{full trajectory-flow+sensitivity model})
=
\mathcal K.
}
\]



## 4. Corollary — Sharp identification and sharp robust risk

Because every point in \(\mathcal K\) is realizable by an actual compatible joint mechanism:

- each LP lower/upper endpoint for \(p_\ell\) is **sharp**;
- for any fixed classifier with leafwise loss vector \(L\),
  \[
  \min_{p\in\mathcal K}L^\top p,\qquad
  \max_{p\in\mathcal K}L^\top p
  \]
  are sharp lower and upper risk bounds;
- no explicit trajectory-flow variables are needed to compute those bounds.

This is stronger than merely showing that the compact LP is conservative.



## 5. Theorem 3 — What multiple horizons can and cannot do

Let \(\mathcal K_S\) denote the compact polytope using a set \(S\) of observation horizons.

If \(S\subseteq S'\), then

\[
\mathcal K_{S'}\subseteq\mathcal K_S.
\]

Therefore additional horizons can **never widen** a sharp identified interval or robust-risk bound.

However, additional horizons need not help. A new horizon is strictly informative exactly when at least one of its new half-spaces is non-redundant.

For a new inequality \(a^\top p\le b\), define

\[
\Delta(a,b;\mathcal K_S)
=
\max_{p\in\mathcal K_S}(a^\top p-b).
\]

Then:

- \(\Delta\le 0\): the inequality is redundant;
- \(\Delta>0\): the new horizon strictly cuts the previous polytope.

Thus strict temporal information has an exact LP characterization. This also explains why some nodes/horizons in earlier experiments produced 0% tightening.


In [1]:

import json, math, time, zipfile
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import linprog

SEED = 20261004
GAMMA = 3.0
H = 3
N_FIXTURES = 24

OUTDIR = Path("selective_refinement_general_tree_theorem_gate_results")
OUTDIR.mkdir(exist_ok=True)

print("Gamma:", GAMMA, "Horizons:", H, "Random fixtures:", N_FIXTURES)


Gamma: 3.0 Horizons: 3 Random fixtures: 24


## 6. Utilities for arbitrary rooted trees

In [2]:

def generate_tree(seed, max_depth=3, min_branch=2, max_branch=3):
    rng = np.random.default_rng(seed)
    children = {}
    counter = [0]

    def rec(node, depth):
        if depth >= max_depth:
            return
        if depth > 0 and rng.random() < 0.18:
            return
        b = int(rng.integers(min_branch, max_branch + 1))
        ch = []
        for _ in range(b):
            counter[0] += 1
            ch.append(f"N{counter[0]}")
        children[node] = ch
        for c in ch:
            rec(c, depth + 1)

    rec("ROOT", 0)
    return children

def prep_tree(children):
    nodes = {"ROOT"}
    parent = {}
    for u, chs in children.items():
        nodes.add(u)
        for c in chs:
            nodes.add(c)
            parent[c] = u

    leaves = sorted(nodes - set(children))

    def path_to_root(node):
        p = [node]
        while p[-1] != "ROOT":
            p.append(parent[p[-1]])
        return list(reversed(p))

    def descendant_leaves(node):
        if node not in children:
            return [node]
        out = []
        for c in children[node]:
            out.extend(descendant_leaves(c))
        return out

    subleaves = {n: descendant_leaves(n) for n in nodes}

    def depth(node):
        d = 0
        while node != "ROOT":
            node = parent[node]
            d += 1
        return d

    return parent, leaves, path_to_root, subleaves, nodes, depth

def assign_valid_edge_curves(children, parent, seed, H=3):
    rng = np.random.default_rng(seed + 1234)
    q = {}
    base = np.linspace(0.25, 0.75, H)

    def rec(u, parent_q=None):
        if u not in children:
            return
        chs = children[u]
        sibling_factor = rng.uniform(0.90, 1.10, len(chs))

        if u == "ROOT":
            for c, f in zip(chs, sibling_factor):
                arr = np.clip(base * f, 0.03, 0.90)
                arr = np.maximum.accumulate(arr)
                q[c] = arr
                rec(c, arr)
        else:
            for c, f in zip(chs, sibling_factor):
                depth_factor = rng.uniform(0.55, 0.78)
                arr = np.minimum(parent_q * depth_factor * f, parent_q * 0.95)
                arr = np.maximum.accumulate(arr)
                q[c] = arr
                rec(c, arr)

    rec("ROOT")
    return q

def max_sibling_odds_ratio(children, q, H=3):
    ans = 1.0
    for u, chs in children.items():
        for h in range(H):
            odds = [q[c][h] / (1.0 - q[c][h]) for c in chs]
            for a in odds:
                for b in odds:
                    ans = max(ans, a / b)
    return ans


## 7. Exact monotone trajectory distributions

In [3]:

def exact_trajectory_distribution(leaves, path_to_root, q, p, H=3):
    group_mass = defaultdict(float)

    for leaf in leaves:
        path = path_to_root(leaf)
        thresholds = sorted(set(
            [0.0, 1.0] +
            [float(q[node][h]) for node in path[1:] for h in range(H)]
        ))

        for a, b in zip(thresholds[:-1], thresholds[1:]):
            if b - a <= 1e-14:
                continue
            U = 0.5 * (a + b)
            states = []
            for h in range(H):
                state = "ROOT"
                for node in path[1:]:
                    if U <= q[node][h] + 1e-15:
                        state = node
                    else:
                        break
                states.append(state)
            group_mass[tuple(states)] += p[leaf] * (b - a)

    assert abs(sum(group_mass.values()) - 1.0) < 1e-10
    return group_mass

def observed_quantities(children, path_to_root, group_mass, H=3):
    r = defaultdict(lambda: defaultdict(lambda: [0.0] * H))
    final_mass = defaultdict(float)

    for g, mass in group_mass.items():
        final_mass[g[-1]] += mass
        for h, state in enumerate(g):
            state_path = set(path_to_root(state))
            for u, chs in children.items():
                for c in chs:
                    if c in state_path:
                        r[u][c][h] += mass

    return r, final_mass


## 8. Compact LP and explicit full-flow LP

In [4]:

def build_compact(children, leaves, subleaves, r, H=3, gamma=3.0, horizons=None):
    if horizons is None:
        horizons = list(range(H))

    idx = {l:i for i,l in enumerate(leaves)}
    n = len(leaves)

    def vec(node):
        v = np.zeros(n)
        for l in subleaves[node]:
            v[idx[l]] = 1.0
        return v

    A, b = [], []

    # Final-node compatibility: s_c >= M_c = r_c,H.
    for u, chs in children.items():
        for c in chs:
            A.append(-vec(c))
            b.append(-r[u][c][H-1])

    # Sibling sensitivity half-spaces.
    for u, chs in children.items():
        vv = {c:vec(c) for c in chs}
        for h in horizons:
            for j in chs:
                for k in chs:
                    if j == k:
                        continue
                    rj, rk = r[u][j][h], r[u][k][h]
                    A.append(rj * vv[k] - gamma * rk * vv[j])
                    b.append((1.0 - gamma) * rj * rk)

    return np.asarray(A), np.asarray(b), idx

def build_full_flow(leaves, path_to_root, group_mass, A_compact, b_compact, idx):
    groups = list(group_mass)
    pairs = []

    for gi, g in enumerate(groups):
        final = g[-1]
        for leaf in leaves:
            if final in path_to_root(leaf):
                pairs.append((gi, leaf))

    P = np.zeros((len(leaves), len(pairs)))
    Aeq = np.zeros((len(groups), len(pairs)))
    beq = np.array([group_mass[g] for g in groups])

    for zi, (gi, leaf) in enumerate(pairs):
        P[idx[leaf], zi] = 1.0
        Aeq[gi, zi] = 1.0

    return groups, pairs, P, Aeq, beq, A_compact @ P, b_compact


## 9. Constructive coupling algorithm from Theorem 1

In [5]:

def greedy_final_node_to_leaf_coupling(
    children, parent, leaves, subleaves, nodes, depth,
    final_mass, pvec, idx, tol=1e-9
):
    residual = {l: float(pvec[idx[l]]) for l in leaves}
    allocation = defaultdict(dict)

    # Deepest final nodes first.
    for v in sorted(nodes, key=depth, reverse=True):
        remaining = float(final_mass.get(v, 0.0))

        for leaf in sorted(subleaves[v]):
            take = min(residual[leaf], remaining)
            if take > 0:
                allocation[v][leaf] = take
                residual[leaf] -= take
                remaining -= take
            if remaining <= tol:
                break

        if remaining > tol:
            return False, allocation, residual

    success = max(abs(x) for x in residual.values()) <= 1e-7
    return success, allocation, residual


## 10. Random-tree theorem stress test

In [6]:

stress_rows = []
global_max_support_difference = 0.0
global_max_coupling_residual = 0.0
all_truth_feasible = True
all_constructive_couplings = True

for seed in range(N_FIXTURES):
    children = generate_tree(seed)
    parent, leaves, path_to_root, subleaves, nodes, depth = prep_tree(children)
    q = assign_valid_edge_curves(children, parent, seed, H)

    max_or = max_sibling_odds_ratio(children, q, H)
    assert max_or <= GAMMA + 1e-10

    rng = np.random.default_rng(seed + 991)
    raw = rng.gamma(2.0, 1.0, len(leaves))
    raw /= raw.sum()
    p_true = dict(zip(leaves, raw))

    group_mass = exact_trajectory_distribution(leaves, path_to_root, q, p_true, H)
    r, final_mass = observed_quantities(children, path_to_root, group_mass, H)

    A, b, idx = build_compact(children, leaves, subleaves, r, H, GAMMA)
    Aeq = np.ones((1, len(leaves)))
    beq = np.array([1.0])
    bounds = [(0, None)] * len(leaves)

    truth = np.array([p_true[l] for l in leaves])
    max_truth_violation = float(max(
        np.max(A @ truth - b),
        abs(truth.sum() - 1.0)
    ))
    all_truth_feasible &= max_truth_violation <= 1e-9

    groups, pairs, P, AeqF, beqF, AF, bF = build_full_flow(
        leaves, path_to_root, group_mass, A, b, idx
    )
    flow_bounds = [(0, None)] * P.shape[1]

    # Compare support functions of compact and full-flow projections.
    rng_obj = np.random.default_rng(seed + 5555)
    max_diff = 0.0
    extreme_points = []

    for k in range(10):
        v = rng_obj.normal(size=len(leaves))
        rc = linprog(
            -v, A_ub=A, b_ub=b, A_eq=Aeq, b_eq=beq,
            bounds=bounds, method="highs"
        )
        rf = linprog(
            -(v @ P), A_ub=AF, b_ub=bF, A_eq=AeqF, b_eq=beqF,
            bounds=flow_bounds, method="highs"
        )
        assert rc.success and rf.success
        max_diff = max(max_diff, abs(rc.fun - rf.fun))
        extreme_points.append(rc.x)

    global_max_support_difference = max(global_max_support_difference, max_diff)

    # Construct a coupling for compact extreme points without using the full-flow LP.
    fixture_constructive = True
    fixture_resid = 0.0
    for pvec in extreme_points[:5]:
        ok, alloc, residual = greedy_final_node_to_leaf_coupling(
            children, parent, leaves, subleaves, nodes, depth,
            final_mass, pvec, idx
        )
        fixture_constructive &= ok
        fixture_resid = max(fixture_resid, max(abs(x) for x in residual.values()))

    all_constructive_couplings &= fixture_constructive
    global_max_coupling_residual = max(global_max_coupling_residual, fixture_resid)

    stress_rows.append({
        "seed":seed,
        "leaves":len(leaves),
        "internal_nodes":len(children),
        "trajectory_groups":len(groups),
        "flow_variables":P.shape[1],
        "compact_inequalities":len(b),
        "max_true_sibling_odds_ratio":max_or,
        "max_truth_violation":max_truth_violation,
        "max_support_function_difference":max_diff,
        "constructive_coupling_success":fixture_constructive,
        "max_constructive_residual":fixture_resid,
    })

stress_df = pd.DataFrame(stress_rows)

assert all_truth_feasible
assert all_constructive_couplings
assert global_max_support_difference < 1e-8
assert global_max_coupling_residual < 1e-7

print(stress_df.to_string(index=False))
print("\nGlobal max support-function discrepancy:", global_max_support_difference)
print("Global max constructive residual:", global_max_coupling_residual)


 seed  leaves  internal_nodes  trajectory_groups  flow_variables  compact_inequalities  max_true_sibling_odds_ratio  max_truth_violation  max_support_function_difference  constructive_coupling_success  max_constructive_residual
    0      18               9                115             188                   176                     1.882249         0.000000e+00                     3.885781e-16                           True               8.326673e-17
    1       8               6                 52              75                    61                     1.325148         1.110223e-16                     4.440892e-16                           True               1.942890e-16
    2      10               7                 64              97                    82                     1.813607         0.000000e+00                     1.942890e-16                           True               0.000000e+00
    3      17              10                107             170                   158  


## 11. Temporal-horizon redundancy audit

The theorem predicts two distinct possibilities:

- earlier horizons may be **redundant**;
- if selection patterns change sufficiently over time, they may cut the final-horizon polytope.

This cell verifies the first possibility over the random suite and then uses a deliberately changing-selection fixture to demonstrate strict tightening.


In [7]:

def leaf_widths(A, b, leaves, idx):
    n = len(leaves)
    eq = np.ones((1,n))
    ans = []

    for leaf in leaves:
        c = np.zeros(n)
        c[idx[leaf]] = 1.0
        lo = linprog(c, A_ub=A, b_ub=b, A_eq=eq, b_eq=[1.0],
                     bounds=[(0,None)]*n, method="highs")
        hi = linprog(-c, A_ub=A, b_ub=b, A_eq=eq, b_eq=[1.0],
                     bounds=[(0,None)]*n, method="highs")
        assert lo.success and hi.success
        ans.append(-hi.fun - lo.fun)

    return np.asarray(ans)

temporal_rows = []

for seed in range(N_FIXTURES):
    children = generate_tree(seed)
    parent, leaves, path_to_root, subleaves, nodes, depth = prep_tree(children)
    q = assign_valid_edge_curves(children, parent, seed, H)

    rng = np.random.default_rng(seed + 991)
    raw = rng.gamma(2.0, 1.0, len(leaves))
    raw /= raw.sum()
    p_true = dict(zip(leaves, raw))

    gm = exact_trajectory_distribution(leaves, path_to_root, q, p_true, H)
    r, _ = observed_quantities(children, path_to_root, gm, H)

    A_last, b_last, idx = build_compact(
        children, leaves, subleaves, r, H, GAMMA, horizons=[H-1]
    )
    A_all, b_all, _ = build_compact(
        children, leaves, subleaves, r, H, GAMMA, horizons=list(range(H))
    )

    w_last = leaf_widths(A_last, b_last, leaves, idx)
    w_all = leaf_widths(A_all, b_all, leaves, idx)

    assert np.all(w_all <= w_last + 1e-8)

    temporal_rows.append({
        "seed":seed,
        "mean_last_width":float(w_last.mean()),
        "mean_all_width":float(w_all.mean()),
        "strictly_tightened_leaves":int(np.sum(w_all < w_last - 1e-7))
    })

temporal_df = pd.DataFrame(temporal_rows)
print(temporal_df.to_string(index=False))


 seed  mean_last_width  mean_all_width  strictly_tightened_leaves
    0         0.095362        0.095362                          0
    1         0.161686        0.161686                          0
    2         0.121241        0.121241                          0
    3         0.104118        0.104118                          0
    4         0.091504        0.091504                          0
    5         0.104733        0.104733                          0
    6         0.128247        0.128247                          0
    7         0.095892        0.095892                          0
    8         0.109442        0.109442                          0
    9         0.130365        0.130365                          0
   10         0.089039        0.089039                          0
   11         0.148596        0.148596                          0
   12         0.099061        0.099061                          0
   13         0.124261        0.124261                          0
   14     

### Deliberate changing-selection fixture

In [8]:

# Reuse the nontrivial depth-3 fixture from the previous Global Compatibility Gate.
children = {
    "ROOT": ["A","B","C"],
    "A": ["A1","A2","A3"],
    "B": ["B1","B2"],
    "C": ["C1","C2","C3"],
    "A1":["A1a","A1b"], "A2":["A2a","A2b"], "A3":["A3a","A3b"],
    "B1":["B1a","B1b"], "B2":["B2a","B2b"],
    "C1":["C1a","C1b"], "C2":["C2a","C2b"], "C3":["C3a","C3b"],
}
parent, leaves, path_to_root, subleaves, nodes, depth = prep_tree(children)

q = {
    "A":np.array([.18,.42,.75]), "B":np.array([.40,.60,.75]), "C":np.array([.28,.50,.75]),
    "A1":np.array([.05,.25,.60]), "A2":np.array([.13,.35,.62]), "A3":np.array([.08,.30,.61]),
    "B1":np.array([.15,.38,.62]), "B2":np.array([.28,.50,.64]),
    "C1":np.array([.09,.30,.60]), "C2":np.array([.19,.42,.63]), "C3":np.array([.12,.34,.61]),
}
for u in list(children):
    if u in ["ROOT","A","B","C"]:
        continue
    base = q[u]
    for j,c in enumerate(children[u]):
        q[c] = base * [0.58,0.82][j]

rng = np.random.default_rng(SEED)
raw = rng.gamma(2.0, 1.0, len(leaves))
raw /= raw.sum()
p_true = dict(zip(leaves, raw))

gm = exact_trajectory_distribution(leaves, path_to_root, q, p_true, H)
r, final_mass = observed_quantities(children, path_to_root, gm, H)

# Gamma=4 for this deliberately more heterogeneous fixture.
fixture_gamma = 4.0
A_last, b_last, idx = build_compact(
    children, leaves, subleaves, r, H, fixture_gamma, horizons=[H-1]
)
A_all, b_all, _ = build_compact(
    children, leaves, subleaves, r, H, fixture_gamma, horizons=list(range(H))
)

w_last = leaf_widths(A_last, b_last, leaves, idx)
w_all = leaf_widths(A_all, b_all, leaves, idx)

strict_gain = float(1.0 - w_all.mean()/w_last.mean())
strict_leaf_count = int(np.sum(w_all < w_last - 1e-7))

assert np.all(w_all <= w_last + 1e-8)
assert strict_leaf_count > 0
assert strict_gain > 0

print("Mean last-horizon width:", w_last.mean())
print("Mean all-horizon width:", w_all.mean())
print("Relative strict gain:", strict_gain)
print("Leaves strictly tightened:", strict_leaf_count, "/", len(leaves))


Mean last-horizon width: 0.11484074798142302
Mean all-horizon width: 0.10966039454569984
Relative strict gain: 0.04510901859121619
Leaves strictly tightened: 12 / 16


## 12. Exact LP redundancy test for new-horizon inequalities

In [9]:

def max_inequality_violation(A_base, b_base, a, beta, n):
    # maximize a^T p - beta over the base polytope
    res = linprog(
        -a,
        A_ub=A_base, b_ub=b_base,
        A_eq=np.ones((1,n)), b_eq=[1.0],
        bounds=[(0,None)]*n,
        method="highs"
    )
    assert res.success
    return float(-res.fun - beta)

# Separate candidate/final constraints + last-horizon sensitivity,
# then test every earlier-horizon sensitivity row for redundancy.
A_candidate, b_candidate, _ = build_compact(
    children, leaves, subleaves, r, H, fixture_gamma, horizons=[]
)
A_last_only, b_last_only, _ = build_compact(
    children, leaves, subleaves, r, H, fixture_gamma, horizons=[H-1]
)
A_all_fixture, b_all_fixture, _ = build_compact(
    children, leaves, subleaves, r, H, fixture_gamma, horizons=list(range(H))
)

base_rows = len(b_last_only)
new_A = A_all_fixture[base_rows:]
new_b = b_all_fixture[base_rows:]

violations = [
    max_inequality_violation(A_last_only, b_last_only, a, beta, len(leaves))
    for a, beta in zip(new_A, new_b)
]

nonredundant = sum(v > 1e-8 for v in violations)
assert nonredundant > 0

redundancy_df = pd.DataFrame({
    "new_constraint_id":np.arange(len(violations)),
    "max_violation_over_last_horizon_polytope":violations,
    "nonredundant":[v > 1e-8 for v in violations]
})

print("Earlier-horizon inequalities tested:", len(violations))
print("Nonredundant inequalities:", nonredundant)
print("Largest cut violation:", max(violations))


Earlier-horizon inequalities tested:

 72
Nonredundant inequalities: 4
Largest cut violation: 0.0003606005369063998


## 13. Save artifacts and issue theorem-gate verdict

In [10]:

stress_df.to_csv(OUTDIR/"random_tree_stress_test.csv", index=False)
temporal_df.to_csv(OUTDIR/"random_tree_temporal_audit.csv", index=False)
redundancy_df.to_csv(OUTDIR/"changing_selection_redundancy_audit.csv", index=False)

plt.figure(figsize=(7,4.5))
plt.scatter(stress_df.leaves, stress_df.flow_variables)
plt.xlabel("Leaves")
plt.ylabel("Full trajectory-flow variables")
plt.title("Full-flow representation size across random trees")
plt.tight_layout()
plt.savefig(OUTDIR/"full_flow_size.png", dpi=160)
plt.close()

plt.figure(figsize=(7,4.5))
plt.scatter(temporal_df.mean_last_width, temporal_df.mean_all_width)
m = max(temporal_df.mean_last_width.max(), temporal_df.mean_all_width.max())
plt.plot([0,m],[0,m])
plt.xlabel("Mean width: last horizon")
plt.ylabel("Mean width: all horizons")
plt.title("Extra horizons never widen random-tree intervals")
plt.tight_layout()
plt.savefig(OUTDIR/"temporal_nonwidening.png", dpi=160)
plt.close()

checks = {
    "projection_theorem_proof_written": True,
    "constructive_sufficiency_proof_written": True,
    "linear_sensitivity_projection_proof_written": True,
    "sharpness_corollary_written": True,
    "temporal_nonwidening_proof_written": True,
    "strict_temporal_gain_characterized_by_lp_redundancy": True,
    "all_random_truths_feasible": bool(all_truth_feasible),
    "all_random_constructive_couplings_succeeded": bool(all_constructive_couplings),
    "random_full_vs_compact_support_functions_match": bool(global_max_support_difference < 1e-8),
    "constructive_coupling_residual_small": bool(global_max_coupling_residual < 1e-7),
    "random_extra_horizons_never_widen": True,
    "changing_selection_fixture_has_strict_temporal_gain": bool(strict_gain > 0),
    "changing_selection_has_nonredundant_earlier_constraints": bool(nonredundant > 0),
}

verdict = (
    "PASS_TO_FORMAL_WRITEUP_AND_NOVELTY_AUDIT"
    if all(checks.values())
    else "STOP_AND_REVISE_THEOREM"
)

summary = {
    "stage":"selective_refinement_general_tree_theorem_gate",
    "fixture_only":False,
    "proof_scope":"arbitrary finite rooted trees with valid monotone taxonomically correct trajectories",
    "gamma_random_suite":GAMMA,
    "horizons":H,
    "random_tree_fixtures":N_FIXTURES,
    "total_random_support_objectives":N_FIXTURES * 10,
    "global_max_full_vs_compact_support_difference":global_max_support_difference,
    "global_max_constructive_coupling_residual":global_max_coupling_residual,
    "max_random_leaves":int(stress_df.leaves.max()),
    "max_random_flow_variables":int(stress_df.flow_variables.max()),
    "changing_selection_relative_multi_horizon_gain":strict_gain,
    "changing_selection_strictly_tightened_leaves":strict_leaf_count,
    "changing_selection_nonredundant_earlier_constraints":int(nonredundant),
    "checks":checks,
    "verdict":verdict,
    "next_step":(
        "Perform a targeted literature novelty audit of the exact theorem: "
        "laminar/tree trajectory-flow projection, sharp subtree-mass sensitivity polytope, "
        "and multi-horizon informative-stopping bounds. If novelty survives, write the formal "
        "propositions/proofs and then add finite-sample uncertainty."
    )
}

with open(OUTDIR/"general_tree_theorem_gate_summary.json","w") as f:
    json.dump(summary,f,indent=2)

zip_name = "selective_refinement_general_tree_theorem_gate_results.zip"
with zipfile.ZipFile(zip_name,"w",zipfile.ZIP_DEFLATED) as zf:
    for p in OUTDIR.iterdir():
        zf.write(p, arcname=p.name)

print(json.dumps(summary, indent=2))
print("\nVERDICT:", verdict)
print("Results archive:", zip_name)

try:
    from google.colab import files
    files.download(zip_name)
except Exception:
    pass


{
  "stage": "selective_refinement_general_tree_theorem_gate",
  "fixture_only": false,
  "proof_scope": "arbitrary finite rooted trees with valid monotone taxonomically correct trajectories",
  "gamma_random_suite": 3.0,
  "horizons": 3,
  "random_tree_fixtures": 24,
  "total_random_support_objectives": 240,
  "global_max_full_vs_compact_support_difference": 1.1102230246251565e-15,
  "global_max_constructive_coupling_residual": 5.238864897449957e-16,
  "max_random_leaves": 21,
  "max_random_flow_variables": 232,
  "changing_selection_relative_multi_horizon_gain": 0.04510901859121619,
  "changing_selection_strictly_tightened_leaves": 12,
  "changing_selection_nonredundant_earlier_constraints": 4,
  "checks": {
    "projection_theorem_proof_written": true,
    "constructive_sufficiency_proof_written": true,
    "linear_sensitivity_projection_proof_written": true,
    "sharpness_corollary_written": true,
    "temporal_nonwidening_proof_written": true,
    "strict_temporal_gain_characteri